In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5")

d:\workspace\AI\langchain-academy\intro-2-langchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ]
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nEstablish and expand a fictional narrative regarding Lunapolis on the moon, focusing on its demographics, environmental conditions, and the active socio-political conflict involving the cheese miners' union and the new president.\n\n## SUMMARY\nSetting: The Moon, Capital identified as Lunapolis. Demographics: 100,000 cheese miners reside in the city. Environment: Clear skies, extreme temperature range of -100C to 120C. Current Conflict: The cheese miners' union is unhappy with the new president and is facing a potential strike. Key Context: The primary narrative focus is resolving the miners' dissatisfaction and the immediate threat of industrial action.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nAddress the root causes of the miners' unhappiness with the new president, evaluate the likelihood of the strike, and explore potential diplomatic solutions or policy changes to mitigate the c

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
Establish and expand a fictional narrative regarding Lunapolis on the moon, focusing on its demographics, environmental conditions, and the active socio-political conflict involving the cheese miners' union and the new president.

## SUMMARY
Setting: The Moon, Capital identified as Lunapolis. Demographics: 100,000 cheese miners reside in the city. Environment: Clear skies, extreme temperature range of -100C to 120C. Current Conflict: The cheese miners' union is unhappy with the new president and is facing a potential strike. Key Context: The primary narrative focus is resolving the miners' dissatisfaction and the immediate threat of industrial action.

## ARTIFACTS
None

## NEXT STEPS
Address the root causes of the miners' unhappiness with the new president, evaluate the likelihood of the strike, and explore potential diplomatic solutions or policy changes to mitigate the conflict.


# Trim/delete messages

- `@before_agent` and `@after_agent` runs once, before or after each run.
- `@before_model` and `@after_model` runs multiple times, before or after each model call

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='c044b52d-9dab-4b7a-838f-152edf11bc54'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='a3b6cca7-e63c-4146-a3df-aa917b93bd6c', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='71fc4665-6a24-4576-a2e6-a8dcf47d37c6'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='df4b4751-4fae-41e4-b3d9-f625273db9de', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='5913521c-1b64-4c04-92bb-2955fd6ceb78'),
              AIMessage(content="I don't have access to your device's internal sensors, so I ca

In [8]:
print(response["messages"][-1].content)

I don't have access to your device's internal sensors, so I cannot tell you the exact temperature. However, extreme temperatures (overheating or freezing) can sometimes prevent a device from turning on.

To check this:
1. **Feel the casing:** Is it uncomfortably hot to the touch or extremely cold?
2. **Look for a power button light:** Even if it won't turn on, does a tiny LED on the charger or power button light up when plugged in?
3. **Force restart:** Try holding the power button for 10–15 seconds to see if that forces it to boot.

If it is too hot, let it cool down for an hour. If you know the device is old or has battery issues, the battery might need replacement. What kind of device is it (phone, laptop, console)? That helps me give more specific advice.
